## Title
Feature Engineering
### By:
Dovaribi Carupia Yagari

### Date:
2026-08-21

### Description:

# Requerimiento

Realizar el proceso de Feature Engineering para limpieza, transformación y modificación de los datos para que puedan ser usados para entrenar un modelo para resolver un problema. Hacerlo en un nuevo branch de git (Usar [Gitflow](https://joserzapata.github.io/courses/ciencia-datos-en-produccion/control-versiones/branching-model/)) y Crear un notebook para la creación de los pipelines de scikit-learn

> Tomar como ejemplo los pasos de: <https://joserzapata.github.io/post/ciencia-datos-proyecto-python/4-feat_eng/>

En este proceso para cada tipo de datos se incluye :

1. **Limpieza de datos**:
  - Eliminar registros datos duplicados (disminuir el numero de datos)
  - Corregir o eliminar valores atípicos (opcional).
  - Los valores atípicos pueden separarse del dataset dependiendo del problema del proyecto (por ejemplo, detección de anomalías).
  - Completar los valores faltantes (por ejemplo, con cero, media, mediana …) o eliminar las filas (o columnas).
2. Selección de atributos (**Feature Selection**) (opcional):
  - Descartar los atributos que no proporcionan información útil para el proyecto.
  - Eliminar registros duplicados (al eliminar atributos pueden quedar registros iguales)
3. Ingeniería de atributos (**Feature Engineering**), cuando sea apropiado:
  - Discretizar las atributos continuas.
  - (opcional) Descomponer en partes los atributos (p. Ej., Categóricas, fecha / hora, etc.).
  - (opcional) Agregar transformaciones prometedoras de las atributos, por ejemplo:
    - log(x)
    - sqrt(x)
    - x^2
    - etc
  - Aplicar funciones a los datos para agregar nuevos atributos.
4. Escalado de atributos (**Feature Scaling**):
  - estandarizar
  - normalizar
  - etc
5. Encoding
  - Encode variables categóricas, texto y las que sean necesarias para poder ser usadas para el modelamiento

Crear todas estas transformaciones usando transformadores y pipelines de scikit-learn
- https://scikit-learn.org/stable/modules/preprocessing.html
- https://scikit-learn.org/stable/modules/compose.html

> Puede utilizar las librerías o herramientas que considere para resolver la tarea

# Entregables

Notebook con la descripción y creación de los pipelines de scikit-learn.

Se debe realizar un Pull request para ingresar el notebook a la rama `main` para esto debe tener mínimo 1 revisión de otras personas del Curso y que pase los checks del CI/CD.

## 1. Objetivo y prevención de data leakage

El objetivo es producir una matriz de características lista para modelamiento sin permitir que información del conjunto de prueba influya en las transformaciones.

Reglas aplicadas en este notebook:

- El target se separa de las características antes de construir el pipeline.
- Las filas sin target se eliminan porque no pueden utilizarse en aprendizaje supervisado.
- Los duplicados exactos se eliminan antes de dividir los datos para evitar que la misma observación aparezca en train y test.
- El split estratificado ocurre antes de ajustar imputadores, transformaciones y escaladores.
- Todos los pasos que aprenden parámetros se ajustan solo con `X_train`; `X_test` se transforma únicamente con esos parámetros.
- Los valores atípicos clínicos no se eliminan automáticamente. Se usa escalado robusto para reducir su influencia numérica sin perderlos.

## 2. Carga y limpieza inicial del dataset

In [1]:
import logging
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PowerTransformer, RobustScaler

In [2]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

DATA_PATH = Path("../..") / "data" / "02_intermediate" / "pacientes_higado_exploracion.parquet"
TARGET = "Diagnosis"

logging.info("Cargando dataset intermedio desde: %s", DATA_PATH)
df = pd.read_parquet(DATA_PATH).copy()

# Normalizamos los tipos esperados sin modificar el archivo Parquet de entrada.
df["Age"] = df["Age"].astype("Int64")
df["Gender"] = df["Gender"].astype("category")
df[TARGET] = df[TARGET].astype("Int64").astype("category")

print(f"Dataset original: {df.shape[0]} filas y {df.shape[1]} columnas")
display(df.dtypes.to_frame("dtype"))
display(df.head())

2026-08-21 09:21:03,028 - INFO - Cargando dataset intermedio desde: ..\..\data\02_intermediate\pacientes_higado_exploracion.parquet


Dataset original: 663 filas y 11 columnas


,dtype
Age,Int64
Gender,category
Total_Bilirubin,float64
Direct_Bilirubin,float64
Alkaline_Phosphotase,float64
Alamine_Aminotransferase,float64
Aspartate_Aminotransferase,float64
Total_Protiens,float64
Albumin,float64
Albumin_and_Globulin_Ratio,float64


,Age,Gender,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Protiens,Albumin,Albumin_and_Globulin_Ratio,Diagnosis
0,65,Female,0.7,0.1,187.0,16.0,18.0,6.8,3.3,0.90,1
1,62,Male,10.9,5.5,699.0,64.0,100.0,7.5,3.2,0.74,1
2,62,Male,7.3,4.1,490.0,60.0,68.0,7.0,3.3,0.89,1
3,58,Male,1.0,0.4,182.0,14.0,20.0,6.8,3.4,1.00,1
4,72,Male,3.9,2.0,195.0,27.0,59.0,7.3,2.4,0.40,1


In [3]:
missing_target = df[TARGET].isna().sum()
duplicate_rows = df.duplicated().sum()

quality_summary = pd.DataFrame(
    {
        "Metric": ["Missing target", "Exact duplicate rows"],
        "Count": [missing_target, duplicate_rows],
        "Percentage": [
            round(missing_target / len(df) * 100, 2),
            round(duplicate_rows / len(df) * 100, 2),
        ],
    }
)
display(quality_summary)

# Estas eliminaciones no aprenden parámetros del dataset y se hacen antes del split.
df_model = df.dropna(subset=[TARGET]).drop_duplicates().copy()
print(f"Dataset después de target faltante y duplicados: {df_model.shape[0]} filas")

,Metric,Count,Percentage
0,Missing target,15,2.26
1,Exact duplicate rows,60,9.05


Dataset después de target faltante y duplicados: 588 filas


## 3. Separación del target y división train/test

Se conserva la interpretación clínica original, pero para modelar se codifica la enfermedad como 1 y la ausencia de enfermedad como 0. Esta recodificación es determinista y no se aprende de los datos.

In [4]:
X = df_model.drop(columns=[TARGET])
y = df_model[TARGET].map({1: 1, 2: 0})

if y.isna().any():
    raise ValueError("El target contiene valores distintos de 1 y 2.")
y = y.astype("int64")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

split_summary = pd.DataFrame(
    {
        "Dataset": ["Train", "Test"],
        "Rows": [len(X_train), len(X_test)],
        "Disease_rate_%": [round(y_train.mean() * 100, 2), round(y_test.mean() * 100, 2)],
    }
)
display(split_summary)
print(f"X_train: {X_train.shape}; X_test: {X_test.shape}")

,Dataset,Rows,Disease_rate_%
0,Train,470,70.85
1,Test,118,71.19


X_train: (470, 10); X_test: (118, 10)


## 4. Atributos derivados dentro del pipeline

Se agregan dos razones clínicas potenciales. Se calculan dentro de un transformador para que el mismo proceso se aplique de forma idéntica en train y test. Los ceros se convierten en `NaN` y serán tratados por el imputador posterior.

In [5]:
class ClinicalFeatureBuilder(BaseEstimator, TransformerMixin):
    """Create deterministic clinical ratios without fitting to the target."""

    def fit(self, X: pd.DataFrame, y: pd.Series | None = None) -> "ClinicalFeatureBuilder":
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        transformed = X.copy()
        total_bilirubin = transformed["Total_Bilirubin"].astype("float64")
        alt = transformed["Alamine_Aminotransferase"].astype("float64")
        transformed["Direct_to_Total_Bilirubin"] = transformed[
            "Direct_Bilirubin"
        ] / total_bilirubin.replace(0, np.nan)
        transformed["AST_to_ALT"] = transformed["Aspartate_Aminotransferase"] / alt.replace(
            0, np.nan
        )
        return transformed

    def get_feature_names_out(self, input_features=None) -> np.ndarray:
        if input_features is None:
            input_features = self.feature_names_in_
        return np.asarray(
            [*input_features, "Direct_to_Total_Bilirubin", "AST_to_ALT"],
            dtype=object,
        )

## 5. Pipelines de imputación, transformación, escalado y encoding

Las variables de laboratorio con sesgo positivo usan transformación Yeo-Johnson y escalado robusto. Las variables numéricas restantes usan imputación y escalado robusto. `Gender` se imputa con la moda y se codifica con one-hot; `handle_unknown="ignore"` permite categorías nuevas en datos futuros.

In [6]:
skewed_numeric_features = [
    "Total_Bilirubin",
    "Direct_Bilirubin",
    "Alkaline_Phosphotase",
    "Alamine_Aminotransferase",
    "Aspartate_Aminotransferase",
    "Direct_to_Total_Bilirubin",
    "AST_to_ALT",
]
regular_numeric_features = [
    "Age",
    "Total_Protiens",
    "Albumin",
    "Albumin_and_Globulin_Ratio",
]
categorical_features = ["Gender"]

skewed_numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("power_transform", PowerTransformer(method="yeo-johnson", standardize=False)),
        ("scaler", RobustScaler()),
    ]
)

regular_numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("skewed_numeric", skewed_numeric_pipeline, skewed_numeric_features),
        ("regular_numeric", regular_numeric_pipeline, regular_numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

feature_pipeline = Pipeline(
    steps=[
        ("clinical_features", ClinicalFeatureBuilder()),
        ("preprocessor", preprocessor),
    ]
)

feature_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('clinical_features', ...), ('preprocessor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('skewed_numeric', ...), ('regular_numeric', ...), ...]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and 

## 6. Ajuste únicamente con train y transformación de test

El método `fit_transform` se aplica solo a `X_train`. Para `X_test` se usa únicamente `transform`, por lo que sus medianas, parámetros de Yeo-Johnson, escalas y categorías no influyen en el aprendizaje de las transformaciones.

In [7]:
X_train_features = feature_pipeline.fit_transform(X_train, y_train)
X_test_features = feature_pipeline.transform(X_test)

feature_names = feature_pipeline.get_feature_names_out()
X_train_features = pd.DataFrame(X_train_features, columns=feature_names, index=X_train.index)
X_test_features = pd.DataFrame(X_test_features, columns=feature_names, index=X_test.index)

print(f"Características generadas: {len(feature_names)}")
print(f"Matriz train transformada: {X_train_features.shape}")
print(f"Matriz test transformada: {X_test_features.shape}")
print(f"Nulos en train transformado: {X_train_features.isna().sum().sum()}")
print(f"Nulos en test transformado: {X_test_features.isna().sum().sum()}")
display(X_train_features.head())

Características generadas: 13
Matriz train transformada: (470, 13)
Matriz test transformada: (118, 13)
Nulos en train transformado: 0
Nulos en test transformado: 0


,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Direct_to_Total_Bilirubin,AST_to_ALT,Age,Total_Protiens,Albumin,Albumin_and_Globulin_Ratio,Gender_Female,Gender_Male
130,0.949045,0.885005,1.317130,0.309190,0.248495,0.581670,-0.015879,-0.04,-0.357143,-0.363636,-0.275,0.0,1.0
571,0.096299,0.000000,0.047086,0.235680,0.749926,-0.193651,1.159720,1.76,0.285714,-0.090909,-0.525,0.0,1.0
93,1.208261,1.148817,-0.059141,1.189230,1.174862,0.684265,0.609075,0.56,0.357143,-0.090909,-0.525,0.0,1.0
104,-0.720136,-0.435379,-0.461558,-0.635072,-0.364954,-0.662308,0.185393,0.68,-0.714286,-0.545455,-0.275,0.0,1.0
329,-0.371629,-0.194044,0.009572,-1.236368,-0.565438,-0.119562,0.475043,-1.00,0.571429,0.909091,0.725,0.0,1.0


In [8]:
fitted_preprocessor = feature_pipeline.named_steps["preprocessor"]
skewed_imputer = fitted_preprocessor.named_transformers_["skewed_numeric"].named_steps["imputer"]
regular_imputer = fitted_preprocessor.named_transformers_["regular_numeric"].named_steps["imputer"]

imputation_summary = pd.DataFrame(
    {
        "Feature": [*skewed_numeric_features, *regular_numeric_features],
        "Train_median_or_value": [
            *skewed_imputer.statistics_,
            *regular_imputer.statistics_,
        ],
    }
)
display(imputation_summary)
print("Los valores mostrados fueron aprendidos exclusivamente desde X_train.")

,Feature,Train_median_or_value
0,Total_Bilirubin,1.000000
1,Direct_Bilirubin,0.300000
2,Alkaline_Phosphotase,210.000000
3,Alamine_Aminotransferase,36.000000
4,Aspartate_Aminotransferase,42.000000
5,Direct_to_Total_Bilirubin,0.307692
6,AST_to_ALT,1.172671
7,Age,46.000000
8,Total_Protiens,6.500000
9,Albumin,3.100000


Los valores mostrados fueron aprendidos exclusivamente desde X_train.


## Conclusiones y decisiones pendientes

- Se eliminaron filas sin target y duplicados exactos antes de la división train/test. Esta decisión evita duplicar observaciones entre conjuntos, pero debe documentarse y revisarse con el origen de los datos.
- Los valores atípicos clínicos se conservaron. `RobustScaler` reduce su influencia en la escala sin convertirlos automáticamente en errores.
- Los valores faltantes se imputan dentro de pipelines. La mediana se calcula solo con train y la moda de `Gender` también se aprende solo con train.
- Se agregaron razones de bilirrubinas y transaminasas como atributos deterministas. Sus posibles beneficios deben confirmarse con validación cruzada en el issue de modelado.
- `Gender` se codifica con one-hot y las categorías desconocidas se ignoran para que el pipeline pueda recibir datos futuros.
- No se aplicó selección supervisada de atributos, balanceo de clases ni evaluación final del modelo. Esas decisiones deben hacerse dentro de una validación cruzada y sin usar el conjunto de prueba para seleccionar transformaciones.

El objeto final `feature_pipeline` es el artefacto de transformación que debe reutilizarse en entrenamiento e inferencia. No se deben recalcular sus parámetros fuera del pipeline ni ajustar transformaciones usando todo el dataset.